Pydantic

In [3]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.5-flash-lite")
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.2.0', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 3.5 Flash Lite', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.5-flash-lite', temperature=None, client=<google.genai.client.Client object at 0x114081400>, default_metadata=(), model_kwargs={})

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title : str = Field(description="Title of the movie")
    year : int = Field(description="Release year of the movie")
    director : str = Field(description="Director of the movie")
    rating : float = Field(description="Movie's rating out of 10")

updated_model = model.with_structured_output(Movie)  
updated_model

updated_model_both = model.with_structured_output(Movie, include_raw=True)



In [5]:
response = updated_model.invoke("Provide details about the movie - Interstellar")
response

response = updated_model_both.invoke("Provide details about the movie - Interstellar")
response



Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'raw': AIMessage(content=[{'type': 'text', 'text': '{\n  "title": "Interstellar",\n  "year": 2014,\n  "director": "Christopher Nolan",\n  "rating": 8.6\n}', 'extras': {'signature': 'El4KXAERTTIP4PD6EVoFkPBRxmuEOf8DImB/hKmb7WZzcu8H6JayoPohc0+ojNJTPL8vqv3I1dtN+UmA9/RUz2eyTb9T9hDb1LY0hj5URrvBP3vk7mhPnj3dI5GW4woB'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a074e2-bdf5-7a83-ba00-f53dd2c948d7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 41, 'total_tokens': 50, 'input_token_details': {'cache_read': 0}}),
 'parsed': Movie(title='Interstellar', year=2014, director='Christopher Nolan', rating=8.6),
 'parsing_error': None}

In [6]:
class Actor(BaseModel):
    name : str = Field("Name of the actor")
    age : int = Field("Age of the actor")

class MovieDetails(BaseModel):
    name : str
    director : str
    cast : list[Actor]

movie_model = model.with_structured_output(MovieDetails)
movie_model.invoke("Provide details about the movie Fight Club")

MovieDetails(name='Fight Club', director='David Fincher', cast=[Actor(name='Edward Norton', age=54), Actor(name='Brad Pitt', age=60), Actor(name='Helena Bonham Carter', age=57)])

Create agent with structured Output

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="gemini-3.5-flash-lite",
    response_format=Movie
)

response = agent.invoke({
    "messages":[{
        "role":"user", "content": "Provide information about Deadpool"
    }]
})

response["structured_response"]

GoogleAuthError: Unable to find your project. Please provide a project ID by:
- Passing a constructor argument
- Using vertexai.init()
- Setting project using 'gcloud config set project my-project'
- Setting a GCP environment variable
- To create a Google Cloud project, please follow guidance at https://developers.google.com/workspace/guides/create-project